# Data Generation

1. **Travel time data**: Sample O/D pairs in Mumbai, call routing API
2. **Pricing data**: Generate synthetic pricing

**Requires**: API on :8000, OSRM on :5001 (for travel time)

## 1. Travel Time Data

In [3]:
import csv
import random
import time
from pathlib import Path

import httpx

MUMBAI_ROAD_COORDS = [
    (19.0596, 72.8341), (19.1136, 72.8491), (19.0760, 72.8777), (18.9388, 72.8354),
    (19.0176, 72.8562), (19.0825, 72.8821), (19.1334, 72.9130), (19.1998, 72.8414),
    (19.0703, 72.8692), (18.9926, 72.8291), (19.0554, 72.8781), (19.0215, 72.8424),
    (19.0896, 72.8656), (19.0027, 72.8025), (19.1688, 72.8591), (19.2193, 72.8378),
    (18.9447, 72.8274), (19.0748, 72.8826), (19.0021, 72.8190),
]

API_BASE = "http://localhost:8000"
NUM_SAMPLES = 1000
out_path = Path("../data/raw/travel_times.csv")
out_path.parent.mkdir(parents=True, exist_ok=True)

rows, failed = [], 0
for i in range(NUM_SAMPLES):
    o = random.choice(MUMBAI_ROAD_COORDS)
    d = random.choice(MUMBAI_ROAD_COORDS)
    if o == d:
        continue
    try:
        r = httpx.get(f"{API_BASE}/route", params={"origin_lat": o[0], "origin_lng": o[1], "dest_lat": d[0], "dest_lng": d[1]}, timeout=30.0)
        r.raise_for_status()
        data = r.json()
    except Exception as e:
        failed += 1
        if failed <= 3:
            print(f"Failed: {e}")
        continue
    rows.append({
        "origin_lat": o[0], "origin_lng": o[1], "dest_lat": d[0], "dest_lng": d[1],
        "distance_km": data["distance_km"], "duration_min": data["duration_min"],
        "hour": random.randint(0, 23), "day_of_week": random.randint(0, 6),
    })
    time.sleep(0.05)
    if (i + 1) % 50 == 0:
        print(f"Done {i + 1}/{NUM_SAMPLES}")

fieldnames = ["origin_lat", "origin_lng", "dest_lat", "dest_lng", "distance_km", "duration_min", "hour", "day_of_week"]
with open(out_path, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=fieldnames)
    w.writeheader()
    w.writerows(rows)

print(f"Saved {len(rows)} rows to {out_path}")
if failed:
    print(f"Failed: {failed}")

Done 50/1000
Done 100/1000
Done 150/1000
Done 200/1000
Done 300/1000
Done 350/1000
Done 400/1000
Done 450/1000
Done 500/1000
Done 550/1000
Done 600/1000
Done 650/1000
Done 700/1000
Done 750/1000
Done 800/1000
Done 850/1000
Done 900/1000
Done 950/1000
Done 1000/1000
Saved 955 rows to ../data/raw/travel_times.csv


## 2. Synthetic Pricing Data

In [4]:
import csv
import random
from pathlib import Path

BASE_PRICE = 50
PER_KM = 12
URGENCY_MULT = {"normal": 1.0, "express": 1.4, "same_day": 1.8}
WEIGHT_PER_KG = 2
VOLUME_PER_L = 0.5

def sample_row():
    distance_km = round(random.uniform(1, 45), 2)
    weight_kg = round(random.uniform(0.5, 25), 1)
    volume_l = round(random.uniform(1, 80), 1)
    urgency = random.choice(list(URGENCY_MULT.keys()))
    hour = random.randint(0, 23)
    day_of_week = random.randint(0, 6)
    demand_score = round(random.uniform(0.5, 1.5), 2)
    price = BASE_PRICE + distance_km * PER_KM + weight_kg * WEIGHT_PER_KG + volume_l * VOLUME_PER_L
    price *= URGENCY_MULT[urgency] * demand_score * random.uniform(0.95, 1.05)
    price = round(max(80, price), 2)
    return {"distance_km": distance_km, "weight_kg": weight_kg, "volume_l": volume_l, "urgency": urgency,
            "hour": hour, "day_of_week": day_of_week, "demand_score": demand_score, "price": price}

rows = [sample_row() for _ in range(5000)]
out_path = Path("../data/raw/pricing_synthetic.csv")
out_path.parent.mkdir(parents=True, exist_ok=True)

with open(out_path, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=rows[0].keys())
    w.writeheader()
    w.writerows(rows)

print(f"Saved {len(rows)} rows to {out_path}")

Saved 5000 rows to ../data/raw/pricing_synthetic.csv
